# Индексы действующих генераторов

Пересобирает два BM25-индекса и три TF-IDF-представления для последней модели. Документы используют только безопасную индексную часть train. Политика и порядок фичей уже сохранены в артефактах. Выполнять только при необходимости пересборки индексов.

## Настройки

In [ ]:
import os

os.environ["OPENBLAS_NUM_THREADS"] = "6"
os.environ["OMP_NUM_THREADS"] = "6"
import gc, re, json, hashlib
from pathlib import Path
from collections import Counter
import numpy as np
import pandas as pd
import bm25s, joblib
from scipy.sparse import load_npz, save_npz
from sklearn.feature_extraction.text import TfidfVectorizer
from tqdm import tqdm
from IPython.display import display

SEED = 42
VALID_LIMIT = 4000
TEST_LIMIT = 10000
BATCH_SIZE = 64
MAX_K = 1000
BUDGETS = [100, 200, 500, 1000]
# Работает при запуске из корня проекта или папки датасета.
PROJECT_DIR = next(
    (folder for folder in [Path.cwd(), *Path.cwd().parents]
     if (folder / "music4all_crs" / "train_queries.parquet").exists()),
    None,
)
if PROJECT_DIR is None:
    raise FileNotFoundError("Не найдена папка music4all_crs с исходными данными")
DATA_DIR = PROJECT_DIR / "music4all_crs"
BASE_DIR = PROJECT_DIR / "retrieval_artifacts"
SEARCH_DIR = BASE_DIR
CACHE_DIR = SEARCH_DIR / "candidate_cache"
OUT_DIR = PROJECT_DIR / "adaptive150_catboost_artifacts"
OUT_DIR.mkdir(exist_ok=True)

from catboost import CatBoostRanker, Pool
from sklearn.feature_extraction.text import CountVectorizer

TRAIN_LIMIT = 30000
ITERATIONS = 350
THREADS = 6
CANDIDATE_LIMIT = 150
POLICY_DIR = OUT_DIR
CAP_CONFIG = json.loads((POLICY_DIR / "candidate_policy.json").read_text())
TOP_K = 1000
RRF_CONSTANT = 60
SOURCE_NAMES = json.loads((SEARCH_DIR / "generators.json").read_text())[
    "generators"
]
OLD_FEATURES = json.loads((BASE_DIR / "base_feature_names.json").read_text())
FEATURES = (
    OLD_FEATURES
    + [f"{name}_source_rank" for name in SOURCE_NAMES]
    + [f"{name}_source_hit" for name in SOURCE_NAMES]
    + ["all_source_count", "all_rrf_score", "selection_rrf_score", "selection_rank"]
)
CAT_FEATURES = ["query_type"]
NUMERIC_FEATURES = [f for f in FEATURES if f not in CAT_FEATURES]
print("Features:", len(FEATURES), "generators:", len(SOURCE_NAMES))

BASE_DIR.mkdir(exist_ok=True)
ARTIFACT_DIR = BASE_DIR


## Безопасное разбиение запросов

In [ ]:
def normalize(text):
    return " ".join(re.findall(r"\w+", str(text).casefold()))


def text_bucket(text):
    return int(hashlib.md5(text.encode()).hexdigest()[:8], 16) % 10


def sample_queries(frame, limit):
    if limit is None or len(frame) <= limit:
        return frame.reset_index(drop=True)
    # Случайная выборка сохраняет естественное распределение типов.
    return frame.sample(n=limit, random_state=SEED).reset_index(drop=True)


train_inputs = pd.read_parquet(DATA_DIR / "train_queries.parquet")
test_inputs = pd.read_parquet(DATA_DIR / "test_public_queries.parquet")
assert train_inputs.query_id.is_unique and test_inputs.query_id.is_unique
train_inputs["normalized"] = train_inputs["query"].map(normalize)
test_texts = set(test_inputs["query"].map(normalize))
buckets = train_inputs.normalized.map(text_bucket)
index_inputs = train_inputs[
    (buckets < 6) & ~train_inputs.normalized.isin(test_texts)
].copy()
rank_inputs = sample_queries(train_inputs[(buckets >= 6) & (buckets < 9)], TRAIN_LIMIT)
valid_inputs = sample_queries(train_inputs[buckets == 9], VALID_LIMIT)
test_inputs = sample_queries(test_inputs, TEST_LIMIT)
assert not set(index_inputs.normalized) & set(rank_inputs.normalized)
assert not set(index_inputs.normalized) & set(valid_inputs.normalized)
assert not set(rank_inputs.normalized) & set(valid_inputs.normalized)
print(
    {
        "index_queries": len(index_inputs),
        "ranker_train": len(rank_inputs),
        "validation": len(valid_inputs),
        "public_test": len(test_inputs),
    }
)
display(test_inputs.query_type.value_counts().rename("test_queries").to_frame())


## Документы каталога

In [ ]:
META_COLUMNS = [
    "m4a_id",
    "m4a_artist",
    "m4a_song",
    "m4a_album",
    "m4a_genres_full",
    "m4a_tags_full",
    "pseudo_caption",
    "lyrics",
    "artist_country",
    "lang",
    "release_year",
    "spotify_popularity",
    "danceability",
    "energy",
    "valence",
    "tempo",
    "duration_ms",
]
tracks = pd.read_parquet(DATA_DIR / "tracks_meta.parquet", columns=META_COLUMNS)
tracks = tracks.sort_values("m4a_id").reset_index(drop=True)
assert tracks.m4a_id.is_unique
track_ids = tracks.m4a_id.to_numpy(dtype=str)
id_to_idx = {track_id: i for i, track_id in enumerate(track_ids)}
train_qrels = pd.read_parquet(DATA_DIR / "train_qrels.parquet")
index_rows = index_inputs.merge(
    train_qrels[["query_id", "target_m4a_id"]], on="query_id", validate="one_to_one"
)
query_docs = (
    index_rows.drop_duplicates(["target_m4a_id", "normalized"])
    .groupby("target_m4a_id")["query"]
    .agg(" ".join)
)
tracks["query_document"] = tracks.m4a_id.map(query_docs).fillna("")
TEXT_COLUMNS = [
    "m4a_artist",
    "m4a_song",
    "m4a_album",
    "m4a_genres_full",
    "m4a_tags_full",
    "pseudo_caption",
    "lyrics",
    "artist_country",
    "lang",
    "release_year",
    "query_document",
]
documents = tracks[TEXT_COLUMNS].fillna("").astype(str).agg(" ".join, axis=1).tolist()
print(
    f"Треков: {len(tracks):,}; с train-запросами: {(tracks.query_document != '').sum():,}"
)
del index_inputs, train_inputs, query_docs
gc.collect()
artists = tracks.m4a_artist.fillna("").map(normalize).to_numpy()
popularity = (
    pd.to_numeric(tracks.spotify_popularity, errors="coerce").fillna(-1).to_numpy()
)
embedding_table = (
    pd.read_parquet(DATA_DIR / "track_embeddings.parquet")
    .set_index("m4a_id")
    .reindex(track_ids)
)
embeddings = np.stack(embedding_table.muq_embedding).astype(np.float32)
embeddings /= np.maximum(np.linalg.norm(embeddings, axis=1, keepdims=True), 1e-8)
del embedding_table
metadata_documents = (
    tracks[[c for c in TEXT_COLUMNS if c not in ["lyrics", "query_document"]]]
    .fillna("")
    .astype(str)
    .agg(" ".join, axis=1)
    .tolist()
)
query_documents = tracks.query_document.tolist()


## BM25 по полным документам

In [ ]:
retriever = bm25s.BM25(k1=1.5, b=0.75, method="lucene")
corpus_tokens = bm25s.tokenize(documents, stopwords="en", show_progress=True)
retriever.index(corpus_tokens, show_progress=True)
retriever.save(str(ARTIFACT_DIR / "bm25_index"))
np.save(ARTIFACT_DIR / "track_ids.npy", track_ids, allow_pickle=False)
del corpus_tokens
gc.collect()

## TF-IDF по полным документам

In [ ]:
tfidf_vectorizer = TfidfVectorizer(
    stop_words="english", ngram_range=(1, 2), sublinear_tf=True,
    max_features=150_000, dtype=np.float32, norm="l2",
)
tfidf_matrix = tfidf_vectorizer.fit_transform(documents).tocsr()
tfidf_transpose = tfidf_matrix.T.tocsr()
joblib.dump(tfidf_vectorizer, ARTIFACT_DIR / "tfidf_vectorizer.joblib")
save_npz(ARTIFACT_DIR / "tfidf_tracks.npz", tfidf_matrix)
print("TF-IDF:", tfidf_matrix.shape, "nonzero:", tfidf_matrix.nnz)
gc.collect()

## Метаданные и train-запросы

In [ ]:
short_bm25 = bm25s.BM25(k1=1.5, b=0.3, method="lucene")
short_tokens = bm25s.tokenize(metadata_documents, stopwords="en", show_progress=False)
short_bm25.index(short_tokens, show_progress=False)
short_bm25.save(str(BASE_DIR / "metadata_bm25_index"))
del short_tokens
query_vectorizer = TfidfVectorizer(
    stop_words="english",
    ngram_range=(1, 2),
    sublinear_tf=True,
    max_features=150000,
    dtype=np.float32,
)
query_matrix = query_vectorizer.fit_transform(query_documents).tocsr()
char_vectorizer = TfidfVectorizer(
    analyzer="char_wb",
    ngram_range=(3, 5),
    min_df=3,
    max_features=120000,
    sublinear_tf=True,
    dtype=np.float32,
)
char_matrix = char_vectorizer.fit_transform(metadata_documents).tocsr()
for name, vectorizer, matrix in [
    ("query_tfidf", query_vectorizer, query_matrix),
    ("metadata_char", char_vectorizer, char_matrix),
]:
    joblib.dump(vectorizer, BASE_DIR / f"{name}_vectorizer.joblib")
    save_npz(BASE_DIR / f"{name}_tracks.npz", matrix)
    print(name, matrix.shape, "nnz:", matrix.nnz)
